## Initial Design

main components for course recommendation system

1. Eligibility_pipeline - Based on the student academic records such as school, competetive exams, UG / PG records etc.. data will be used to check eligibility from the courses that university provides.

2. Similarity_search_model - cosine similarity engine to search for relevant courses among the eligible courses. Then rank the Top-K courses as the output


## Current action plan

First the course data will be required in the structured format to store it as vector embedding, which will be used for similarity search.

Course data will be structured as follows:
```python
1. course_id
2. course_name
3. degree_level        # UG / PG / PhD
4. domain              # Engineering / Management /5. Science / Humanities

eligible_degrees    # list-like string
required_subjects   # list-like string

required_exam       # JEE / CAT / GATE / NET / NONE
min_exam_score
min_gpa

description         # for TF-IDF
keywords            # comma separated
career_outcomes     # optional but useful
```


## First Implementation

The first code implemetation originated from the need of a clean dataset for the eligibility pipeline. 

The current dataset had the following fields.
1. Program Name
2. Program level
3. Name of Faculty
4. Eligibility criteria

There were more fields in the datset but for the project's scope these are relevant.

The restructuring pipeline [restructure.ipynb](../../restructure_pipeline/restructure.ipynb) has the complete implementation.
The pipeline is a small Agentic workflow made with langchain-ollama library uses a locally running model `llama3.2` to generate a structure output which then will be handled by the `pandas` library to generate the desired dataset.

The ouput dataset will have the following fields
1. program_name: str 
2. program_level: str 
3. domain: str 
4. eligibility: str 
5. description: str 
6. skills_learned: List[str]
7. career_outcomes: List[str] 

The structure output is genrate by the model using pydantic model to ensure type safety. Dateset with these dimensions will be used for eligibility criteria and ML similarity to generate recommendations.

## Eligibility Pipeline

The dataset has been finalised and is ready for eligibility pipeline. This pipeline will serve the purpose of filteriing all the eligible course for a student profile. The filtered courses then will be matched against the student's profile in a cosine similarity engine.

The finalised dataset has the following schema:-
```python
class EligibilityStruct(BaseModel):
    min_degree_level: Optional[str] = Field(
        default=None,
        description="One of: PreUG, UG, PG, PhD"
    )
    min_marks_general: Optional[float] = Field(
        default=None,
        description="Minimum percentage for general category",
        ge=0.0,
        le=100.0
    )
    min_marks_reserved: Optional[float] = Field(
        default=None,
        description="Minimum percentage for reserved category (SC/ST/OBC/PwD)",
        ge=0.0,
        le=100.0
    )

class DatasetRow(BaseModel):
    program_name: str = Field(description="Official name of the academic program")
    program_level: str = Field(description="UG, PG, Diploma, Certificate, or PhD level")
    domain: str = Field(description="Broad academic domain such as Engineering, Science, Management, Humanities, etc.")
    eligibility: str = Field(description="Eligibility criteria required for admission")
    description: str = Field(description="Short academic description of the program")
    skills_learned: List[str] = Field(
        description="Key academic or professional skills gained after completing the program"
    )
    career_outcomes: List[str] = Field(
        description="Typical career paths or job roles after completing the program"
    )
    eligibility_struct: EligibilityStruct = Field(description="Eligibiliy in the structured format")
```

The dataset schema is validates using pydantic models ensuring type-safety during runtime.

## Data Flattening script

This script was used for flattening the `eligibility_struct` field as it was causing parsing errors in the eligibility pipeline.

In [8]:
import pandas as pd
import re

def parse_eligibility_struct(text):
    degree = re.search(r"min_degree_level='(.*?)'", text)
    gen = re.search(r"min_marks_general=([\d.]+)", text)
    res = re.search(r"min_marks_reserved=([\d.]+)", text)

    return pd.Series({
        "min_degree_level": degree.group(1) if degree else None,
        "min_marks_general": float(gen.group(1)) if gen else None,
        "min_marks_reserved": float(res.group(1)) if res else None
    })
df = pd.read_csv("final_data2.csv")

elig_cols = df["eligibility_struct"].apply(parse_eligibility_struct)

df = pd.concat([df, elig_cols], axis=1, names="struct_eligibility")
df = df.drop(columns=["Unnamed: 0.1", "Unnamed: 0", "eligibility_struct"])
df.to_csv("final_data3.csv", index=False)
print(f"flattened {len(df)} records")


    


flattened 114 records


### Experiment Log: TF-IDF Based Course Recommendation System

#### Objective

To evaluate the effectiveness of TF-IDF vectorization combined with cosine similarity for recommending academic courses based on student profiles.

---

#### Implementation Summary

* Course features used:

  * Program Name
  * Domain
  * Description
  * Skills Learned
  * Career Outcomes

* Student features used:

  * Academic Background
  * Subjects
  * Interests
  * Preferred Skills
  * Career Goal
  * Preferred Domain

* Pipeline:

  1. TF-IDF vectorizer fitted on all course texts
  2. Student and eligible course texts transformed into vectors
  3. Cosine similarity used to rank courses
  4. Top-K courses selected as recommendations

---

#### Observations

1. **Top-K Quality Distribution**

   * The top 1–2 recommendations are highly relevant
   * Top 3–5 are moderately relevant
   * Beyond top 5, relevance drops significantly

2. **Domain Alignment**

   * Highest-ranked recommendations often align with the student's preferred domain
   * Secondary recommendations align more with skills/interests rather than domain

3. **Quality Degradation Pattern**

   * Ranking quality decreases progressively
   * Lower-ranked results include semantically weak matches or domain mismatches

---

#### Root Cause Analysis

1. **TF-IDF Limitation**

   * Relies on keyword overlap rather than semantic meaning
   * Cannot distinguish contextual differences (e.g., "data analysis" in biology vs computer science)

2. **Feature Imbalance**

   * Long descriptions dominate vector representation
   * Domain signal is underweighted

3. **Dataset Constraints**

   * Limited diversity of courses
   * Overlap of generic keywords across domains introduces noise

---

#### Key Insights

* TF-IDF is effective for **high-confidence top recommendations (Top-1 to Top-3)**
* It fails in **fine-grained ranking beyond top candidates**
* Domain awareness is weak without explicit weighting or filtering
* Suitable as a **baseline model**, not a final solution

---

#### Conclusion

TF-IDF with cosine similarity provides a strong baseline for course recommendation systems, especially for identifying top candidates. However, it lacks semantic understanding and domain sensitivity, leading to reduced effectiveness in deeper ranking.

---

#### Next Direction

* Introduce embedding-based similarity for semantic understanding
* Compare TF-IDF vs embedding performance on ranking quality
* Explore hybrid approaches combining both methods
* Implement evaluation metrics (e.g., Precision@K, domain match rate)
